1) Install packages

In [1]:
!pip install qiskit qiskit-machine-learning scikit-learn matplotlib pandas numpy

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/8.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.6 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.6 MB 648.0 kB/s eta 0:00:13
   -- ------------------------------------- 0.5/8.6 MB 648.0 kB/s eta 0:00:13
   --- ------------------------------------ 0.8/8.6 MB 646.9 kB/s eta 0:00:13
   --- ------------------------------------ 0.8/8.6 MB 646.9 kB/s eta 0:00:13
   --- ------------------------------------ 0.8/8.6 MB 646.9 kB/s eta 0:00:13
   ---- ----------------------------------- 1.0/8.6 MB 577.0 kB/s eta 0:00:14
   ---- ----------------------------------- 1.0/8.6 MB 577.0 kB/s eta 0:00:14
   ---- ----------------------------------- 1.0/8.6 MB 577.0 kB/s eta 0:00:14
   ------ ------------

2) Imports

In [2]:
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.decomposition import PCA
from sklearn.utils import resample

from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

3) Load your processed feature file

In [11]:
df = pd.read_csv("../data/processed/activity_fusion_features.csv")
df.head()

,hr_mean,hr_std,acc_mean,acc_std,acc_energy,gyro_mean,gyro_std,temp_synth,emg_rms_synth,emg_mav_synth,emg_wl_synth,label
0,103.74,2.103455,9.876993,0.856301,98.284573,9.921463,0.883948,36.387673,0.054783,0.047121,0.255089,0
1,103.64,2.660034,9.823884,0.292048,96.593571,9.886520,0.249667,36.353994,0.053765,0.034194,0.304114,0
2,98.40,3.804256,9.793069,0.097488,95.913666,9.856697,0.042707,36.306204,0.051431,0.027732,0.236891,0
3,93.84,1.595975,9.796561,0.093119,95.981233,9.855740,0.039574,36.337672,0.063212,0.054502,0.280872,0
4,92.04,0.749137,9.798123,0.091727,96.011595,9.857234,0.037344,36.370531,0.049137,0.038557,0.382329,0


4) Check columns

In [12]:
print(df.columns.tolist())
print(df["label"].value_counts())

['hr_mean', 'hr_std', 'acc_mean', 'acc_std', 'acc_energy', 'gyro_mean', 'gyro_std', 'temp_synth', 'emg_rms_synth', 'emg_mav_synth', 'emg_wl_synth', 'label']
label
1    126
2     50
0     45
Name: count, dtype: int64


5) Keep only the classes you want

In [13]:
valid_classes = ["rest", "moderate", "intense"]
df = df[df["label"].isin(valid_classes)].copy()

print(df["label"].value_counts())

Series([], Name: count, dtype: int64)


6) Remove non-feature columns

In [14]:
non_feature_cols = ["label", "timestamp"]
feature_cols = [c for c in df.columns if c not in non_feature_cols and pd.api.types.is_numeric_dtype(df[c])]

X = df[feature_cols].copy()
y = df["label"].copy()

print("Number of features:", len(feature_cols))
print(feature_cols)

Number of features: 11
['hr_mean', 'hr_std', 'acc_mean', 'acc_std', 'acc_energy', 'gyro_mean', 'gyro_std', 'temp_synth', 'emg_rms_synth', 'emg_mav_synth', 'emg_wl_synth']


7) Balance the classes

In [17]:
import pandas as pd

df = pd.read_csv("../data/processed/activity_fusion_features.csv")

print(df.shape)
print(df.columns.tolist())
print(df["label"].value_counts(dropna=False))
print(df["label"].unique())

(221, 12)
['hr_mean', 'hr_std', 'acc_mean', 'acc_std', 'acc_energy', 'gyro_mean', 'gyro_std', 'temp_synth', 'emg_rms_synth', 'emg_mav_synth', 'emg_wl_synth', 'label']
label
1    126
2     50
0     45
Name: count, dtype: int64
[0 1 2]


In [15]:
df_balanced = []

min_count = df["label"].value_counts().min()
print("Samples per class after balancing:", min_count)

for cls in df["label"].unique():
    cls_df = df[df["label"] == cls]
    cls_down = resample(cls_df, replace=False, n_samples=min_count, random_state=42)
    df_balanced.append(cls_down)

df_balanced = pd.concat(df_balanced).sample(frac=1, random_state=42).reset_index(drop=True)

X = df_balanced[feature_cols].copy()
y = df_balanced["label"].copy()

print(df_balanced["label"].value_counts())

Samples per class after balancing: nan


ValueError: No objects to concatenate